# 04 Model Evaluation & SHAP Explainability
**Smart India Hackathon 2026 — Problem Statement 26102**

This notebook demonstrates:
1. **Genuine SHAP (SHapley Additive exPlanations)** using `shap.TreeExplainer` on trained tree models.
2. **Calibration & Risk Score Banding**.
3. **Comparative Case Studies**: Verifying that a clean safe project receives low risk (< 35) while fraudulent, duplicated, or prohibited projects receive appropriate escalations.

In [ ]:
import sys
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

sys.path.append(str(Path.cwd().parent))
from src.config import MODELS_DIR
from src.backend.services.risk_scorer import RiskScorerService
from src.explainability.shap_explainer import compute_shap_explanations

scorer = RiskScorerService()
print("Initialized RiskScorerService successfully.")

## 1. Case Study 1: Clean, Compliant, On-Time Public Work
₹10 Lakhs solar street light installation in Varanasi, 90% physically complete, 0 days behind schedule, zero contractor overruns.

In [ ]:
safe_project = {
    "project_id": "CS_SAFE_01",
    "project_name": "Installation of Solar Street Lights in Gram Panchayat",
    "category": "Energy",
    "amount_sanctioned": 1000000.0,
    "amount_spent": 850000.0,
    "progress_percentage": 90.0,
    "approval_date": "2023-01-01",
    "expected_completion_date": "2023-10-01",
    "actual_completion_date": "2023-09-15",
    "district": "Varanasi",
    "state": "Uttar Pradesh",
    "contractor_name": "UP Power Grid Ltd",
    "previous_contractor_projects": 12,
    "previous_contractor_overruns": 0
}

res_safe = scorer.analyze_single_project(safe_project)
print(f"Risk Score: {res_safe['risk_score']}/100 ({res_safe['risk_category'].upper()} RISK)")
print(f"Anomaly Score: {res_safe['anomaly_score']:.4f}")
print(f"Fraud Probability: {res_safe['fraud_risk']['probability']:.4f}")
print(f"Efficiency Score: {res_safe['efficiency_risk']['score']:.4f}")
print(f"Compliance Status: {res_safe['compliance_assessment']['overall_status']}")
print(f"Alert Escalation: {res_safe['alert_escalation']}")

## 2. Case Study 2: Suspicious Work (Extreme Overrun & Contractor Concurrency)
₹50 Lakhs sanctioned, ₹95 Lakhs spent (+90% overrun), contractor handling 8 concurrent public sites with 4 past overruns.

In [ ]:
suspect_project = {
    "project_id": "CS_SUSPECT_02",
    "project_name": "Construction of Commercial Market Complex",
    "category": "Roads & Bridges",
    "amount_sanctioned": 5000000.0,
    "amount_spent": 9500000.0,  # 90% overrun
    "progress_percentage": 40.0,
    "approval_date": "2022-01-01",
    "expected_completion_date": "2023-01-01",
    "days_behind_schedule": 240,
    "district": "Hingoli",
    "state": "Maharashtra",
    "contractor_name": "Nexus Developers",
    "previous_contractor_projects": 8,
    "previous_contractor_overruns": 4
}

res_suspect = scorer.analyze_single_project(suspect_project)
print(f"Risk Score: {res_suspect['risk_score']}/100 ({res_suspect['risk_category'].upper()} RISK)")
print(f"Alert Escalation: {res_suspect['alert_escalation']}")
print("Top SHAP Drivers:")
for d in res_suspect['shap_explanations']['primary_contributors'][:4]:
    print(f" - {d['display_name']}: +{d['impact_points']} pts ({d['direction']})")

## 3. Case Study 3: Prohibited Works Detection (MoSPI MPLADS Policy)
A project attempting to fund a private religious prayer hall, prohibited under MPLADS guidelines.

In [ ]:
prohibited_project = {
    "project_id": "CS_PROHIBITED_03",
    "project_name": "Construction of Temple Mandir and Religious Ashram",
    "category": "Religious Structure",
    "amount_sanctioned": 1500000.0,
    "amount_spent": 500000.0,
    "progress_percentage": 30.0,
    "approval_date": "2023-01-01",
    "expected_completion_date": "2023-12-01",
    "district": "Varanasi",
    "state": "Uttar Pradesh",
    "work_description": "Construction of prayer hall inside mandir ashram premises."
}

res_proh = scorer.analyze_single_project(prohibited_project)
print(f"Compliance Status: {res_proh['compliance_assessment']['overall_status']}")
print(f"Compliance Score: {res_proh['compliance_assessment']['compliance_score']}/100")
print("Violations:")
for v in res_proh['compliance_assessment']['violations']:
    print(f" - [{v['rule_id']}]: {v['message']}")